In [25]:
# Run once to install the interactive backend
%pip install -q ipympl

Note: you may need to restart the kernel to use updated packages.


In [26]:
%matplotlib qt

In [27]:
import npoly3d as n3
import numpy as np
import pinocchio as pin

import yaml
import sympy as sym
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [28]:
N = 100

_gamma = np.zeros((N, 6))
_gamma[:, 2] = np.linspace(0, 1, N)
gamma = np.array([np.array(pin.exp(gi)) for gi in _gamma])

sfx = ''
if sfx == '':
    gab = np.diag([1, 1, 1, 0, 0, 0])
elif sfx == '1':
    gab = np.diag([1, 1, 1, 1, 1, 1])

sx = sym.symbols('t')
with open(f'bases_fns/_orthopoly{sfx}.yaml', 'r') as f:
    polys_string = yaml.safe_load(f)
pp = [sym.sympify(ps) for ps in polys_string]
print('loaded bases')

def eval_poly(x):
    return np.array([pi.subs({sx: x}) for pi in pp]).astype(float)

tt = np.linspace(0, 1, N)
npoly = np.array([eval_poly(ti) for ti in tt])

loaded bases


In [29]:
xi1 = np.zeros((N, 6))
xi1[:, 3] = npoly[:, 0]   # omega_x, first poly

xi2 = np.zeros((N, 6))
xi2[:, 4] = npoly[:, 1]   # omega_y, second poly

xi3 = np.zeros((N, 6))
xi3[:, 5] = npoly[:, 0]   # omega_z, first poly

xis = [xi1, xi2, xi3]

def metric(c1, c2, c3):
    gamma_c = n3.shape_exp(c1 * xi1 + c2 * xi2 + c3 * xi3)
    G = np.zeros((3, 3))
    for i in range(3):
        for j in range(i, 3):
            G[i, j] = n3.inner_product(gamma_c, gab, xis[i], xis[j])
            G[j, i] = G[i, j]
    return G

In [30]:
print("Metric at origin:")
print(metric(0, 0, 0))
print()
print("Metric at (0.3, -0.2, 0.1):")
print(metric(0.3, -0.2, 0.1))

Metric at origin:
[[1.00017005 0.         0.        ]
 [0.         1.0013809  0.        ]
 [0.         0.         0.        ]]

Metric at (0.3, -0.2, 0.1):
[[ 0.30877105  0.03678328  0.11934484]
 [ 0.03678328  2.11706168 -0.12380183]
 [ 0.11934484 -0.12380183  0.14252588]]


In [31]:
def draw_ellipsoid(cov, center, ax, n_std=0.05, color='b', alpha=0.4):
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals = vals[order]
    vecs = vecs[:, order]

    radii = n_std * np.sqrt(vals)

    u = np.linspace(0, 2 * np.pi, 20)
    v = np.linspace(0, np.pi, 14)
    sx = np.outer(np.cos(u), np.sin(v))
    sy = np.outer(np.sin(u), np.sin(v))
    sz = np.outer(np.ones_like(u), np.cos(v))

    sphere = np.stack([sx, sy, sz], axis=-1)
    ellipsoid = sphere * radii[None, None, :]
    ellipsoid = ellipsoid @ vecs.T

    ex = ellipsoid[..., 0] + center[0]
    ey = ellipsoid[..., 1] + center[1]
    ez = ellipsoid[..., 2] + center[2]

    ax.plot_surface(ex, ey, ez, color=color, alpha=alpha, linewidth=0)

In [32]:
# fig = plt.figure(figsize=(8, 8))
# ax = fig.add_subplot(projection='3d')

# xx = np.linspace(-0.5, 0.5, 9)
# coords = []
# for i in range(len(xx)):
#     for j in range(len(xx)):
#         for k in range(len(xx)):
#             c1, c2, c3 = xx[i], xx[j], xx[k]
#             coords.append((c1, c2, c3))
#             draw_ellipsoid(metric(c1, c2, c3), (c1, c2, c3), ax, .05)

# coords = np.array(coords)
# ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], s=1, c='k')
# ax.set_xlabel('c1')
# ax.set_ylabel('c2')
# ax.set_zlabel('c3')
# ax.set_title('Metric Ellipsoids')
# try:
#     ax.set_aspect('equal')
# except NotImplementedError:
#     ax.set_box_aspect([1, 1, 1])
# plt.tight_layout()
# plt.show()

In [33]:
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(projection='3d')

xx = np.linspace(-0.5, 0.5, 5)
coords = []
for i in range(len(xx)):
    for j in range(len(xx)):
        for k in range(len(xx)):
            c1, c2, c3 = xx[i], xx[j], xx[k]
            coords.append((c1, c2, c3))
            draw_ellipsoid(metric(c1, c2, c3), (c1, c2, c3), ax, .05)

coords = np.array(coords)
ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], s=1, c='k')
ax.set_xlabel('c1')
ax.set_ylabel('c2')
ax.set_zlabel('c3')
ax.set_title('Metric Ellipsoids (sparse)')
try:
    ax.set_aspect('equal')
except NotImplementedError:
    ax.set_box_aspect([1, 1, 1])
plt.tight_layout()
plt.show()

In [34]:
# gam = n3.shape_exp(0.5 * xi1 - 0.8 * xi2 + 0.3 * xi3)

# fig = plt.figure(figsize=(6, 6))
# ax = fig.add_subplot(projection='3d')
# ax.plot(gam[:, 0, 3], gam[:, 1, 3], gam[:, 2, 3], label='bent arm')
# ax.scatter(0, 0, 0, c='r', s=30)
# ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
# try:
#     ax.set_aspect('equal')
# except NotImplementedError:
#     ax.set_box_aspect([1, 1, 1])
# ax.legend()
# ax.set_title('Arm shape: 0.5*xi1 - 0.8*xi2 + 0.3*xi3')
# plt.show()

In [35]:
# metric(0, .1, 0)

In [36]:
# fig = plt.figure()
# ax = fig.add_subplot(projection='3d')
# ax.plot(gamma[:, 0, 3], gamma[:, 1, 3], gamma[:, 2, 3], '.-', markersize=2)
# ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
# try:
#     ax.set_aspect('equal')
# except NotImplementedError:
#     ax.set_box_aspect([1, 1, 1])
# ax.set_title('Straight reference arm')
# plt.show()